# 04 — Model Comparison & Deployment Discussion

Loads the saved metrics/reports from `02_scratch_cnn.ipynb` and `03_transfer_learning.ipynb` and compares them:
overall accuracy, per-class F1 (imbalance effect), model size, and inference speed — to answer the brief's
final question: *which model is better for real-time deployment?*

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BACKBONE = "resnet50"  # must match what you used in 03_transfer_learning.ipynb

with open("artifacts/scratch_cnn_metrics.json") as f:
    scratch_metrics = json.load(f)
with open(f"artifacts/{BACKBONE}_metrics.json") as f:
    pretrained_metrics = json.load(f)

scratch_report = pd.read_csv("artifacts/scratch_cnn_classification_report.csv", index_col=0)
pretrained_report = pd.read_csv(f"artifacts/{BACKBONE}_classification_report.csv", index_col=0)

CLASSES = ["Normal", "Diabetic Retinopathy", "Others", "Glaucoma",
           "Cataract", "Myopia", "AMD", "Hypertension"]

## Headline comparison

In [ ]:
comparison = pd.DataFrame([scratch_metrics, pretrained_metrics]).set_index("model")
comparison["throughput_fps"] = 1000 / comparison["ms_per_image"]
comparison[["test_accuracy", "macro_f1", "weighted_f1", "ms_per_image", "throughput_fps", "n_params"]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

comparison[["test_accuracy", "macro_f1"]].plot(kind="bar", ax=axes[0])
axes[0].set_title("Accuracy / Macro-F1"); axes[0].set_ylim(0, 1); axes[0].tick_params(axis='x', rotation=20)

comparison["ms_per_image"].plot(kind="bar", ax=axes[1], color="orange")
axes[1].set_title("Latency (ms/image, lower=better)"); axes[1].tick_params(axis='x', rotation=20)

comparison["n_params"].plot(kind="bar", ax=axes[2], color="green")
axes[2].set_title("Parameter count"); axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig("artifacts/model_comparison_overview.png", dpi=150)
plt.show()

## Per-class F1 — effect of class imbalance

In [ ]:
per_class = pd.DataFrame({
    "scratch_cnn_f1": scratch_report.loc[CLASSES, "f1-score"],
    "pretrained_f1": pretrained_report.loc[CLASSES, "f1-score"],
    "support": scratch_report.loc[CLASSES, "support"],
}).sort_values("support", ascending=False)
per_class

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
per_class[["scratch_cnn_f1", "pretrained_f1"]].plot(kind="bar", ax=ax)
ax.set_title("Per-class F1: Scratch CNN vs Fine-tuned Pretrained Model\n(sorted by training support, left=most data)")
ax.set_ylabel("F1-score"); ax.set_xticklabels(per_class.index, rotation=35, ha="right")
plt.tight_layout()
plt.savefig("artifacts/per_class_f1_comparison.png", dpi=150)
plt.show()

## Discussion (fill in with your actual numbers after running 02/03)

**Effect of class imbalance on per-class performance**
- Both models are expected to score highest F1 on `Normal` and `Diabetic Retinopathy` (most training data) and
  lowest on `Hypertension`, `AMD`, `Myopia` (fewest examples, ~30–110 images each).
- Class weighting narrows this gap versus unweighted training, but does not eliminate it — with only a handful of
  validation/test images per rare class, recall on those classes stays volatile.
- The **scratch CNN** typically overfits faster on minority classes (memorizes the few examples) while generalizing
  worse than the pretrained model, because it must learn low-level visual features from scratch on a relatively
  small, imbalanced dataset.
- The **pretrained model** (ResNet50/VGG16 fine-tuned) starts from ImageNet features (edges, textures, shapes)
  that transfer well to fundus images, so it typically needs less data per class to reach a usable F1 on rare classes.

**Which model is better for real-time deployment?**
Trade-off to weigh explicitly once you have numbers:
- **Accuracy**: the fine-tuned pretrained model is expected to outperform the scratch CNN on macro-F1 (per the
  completion requirement that the pretrained model should outperform the scratch CNN).
- **Speed / footprint**: the scratch CNN is smaller and typically faster per inference (fewer parameters, no
  deep residual blocks), which matters for real-time or edge deployment (e.g. a clinic device with modest hardware).
- **Recommendation template**: *"For a clinical screening context where missing a rare pathology (e.g. Glaucoma,
  AMD) is costly, prioritize the fine-tuned pretrained model's higher recall despite extra latency. For a
  resource-constrained real-time triage tool where an initial fast pass is followed by human review, the smaller
  scratch CNN's speed may be preferable, or consider knowledge-distilling the pretrained model into a smaller
  network / using a lighter backbone (MobileNetV2/EfficientNet-Lite) to get both speed and accuracy."*
- Replace this paragraph with your actual measured accuracy/F1/latency numbers before submitting.

In [ ]:
comparison.to_csv("artifacts/model_comparison_summary.csv")
print("Saved artifacts/model_comparison_summary.csv")